# Geographic map of author affiliations
Reads `data/raw/icsa_affiliations.json`, counts rows by country, and saves a choropleth to `figures/`.

In [1]:
# pip install geopandas matplotlib mapclassify

In [2]:
import json
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap, Normalize
import pandas as pd
import geodatasets
from shapely.geometry import MultiPolygon




## 1. Config — change these for a different conference

In [3]:
INPUT_FILE  = Path("../../data/raw/icsa_affiliations.json")
OUTPUT_FILE = Path("figures/geo_map.png")
TITLE       = "ICSA / WICSA / ECSA (2010–2025)"

# Optional: restrict to specific years. Set to None to include all.
YEARS = None          # e.g. [2020, 2021, 2022]

# Colormap — any matplotlib sequential cmap works: 'Blues', 'YlOrRd', 'Purples'…
CMAP   = LinearSegmentedColormap.from_list(
    "white_darkred",
    ["#fff5f0", "#fcbba1", "#fc9272", "#fb6a4a", "#de2d26", "#a50f15", "#67000d"],
)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

## 2. Load & count

In [4]:
with open(INPUT_FILE) as f:
    records = json.load(f)

df = pd.DataFrame(records)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["country_code"] = df["country_code"].str.upper().str.strip()

if YEARS:
    df = df[df["year"].isin(YEARS)]
    print(f"Filtered to {YEARS}: {len(df):,} rows")
else:
    print(f"All years: {len(df):,} rows")

counts = (
    df.dropna(subset=["country_code"])
    .groupby("country_code")
    .size()
    .reset_index(name="count")
)

total = counts["count"].sum()
counts["pct"] = (counts["count"] / total * 100).round(1)
counts.sort_values("count", ascending=False).head(10)

All years: 1,943 rows


,country_code,count,pct
10,DE,268,14.7
38,US,244,13.3
15,FR,199,10.9
2,AU,161,8.8
26,NL,126,6.9
22,IT,123,6.7
32,SE,107,5.9
1,AT,96,5.3
4,BR,62,3.4
16,GB,57,3.1


## 3. Load world geometry & merge

In [5]:
url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
world_full = gpd.read_file(url)

SOVEREIGN_TYPES = {"Sovereign country", "Country"}
world = world_full[world_full["TYPE"].isin(SOVEREIGN_TYPES)].copy()

# keep "ADMIN" as "name" for the France fix, drop it after
world = world.rename(columns={"ADM0_A3": "iso_a3", "ADMIN": "name"})[["iso_a3", "name", "geometry"]]

# fix France — remove French Guiana
france_geoms = list(world.loc[world["name"] == "France", "geometry"].values[0].geoms)
mainland = [g for g in france_geoms if g.centroid.x > -30]
world.loc[world["name"] == "France", "geometry"] = MultiPolygon(mainland)

world = world.drop(columns=["name"])

# build alpha-2 → alpha-3 lookup
try:
    import pycountry
    def a2_to_a3(a2):
        try:
            return pycountry.countries.get(alpha_2=a2).alpha_3
        except Exception:
            return None
    counts["iso_a3"] = counts["country_code"].apply(a2_to_a3)

except ImportError:
    A2_TO_A3 = {
        "AF":"AFG","AL":"ALB","DZ":"DZA","AO":"AGO","AR":"ARG","AM":"ARM",
        "AU":"AUS","AT":"AUT","AZ":"AZE","BE":"BEL","BR":"BRA","BG":"BGR",
        "CA":"CAN","CL":"CHL","CN":"CHN","CO":"COL","HR":"HRV","CZ":"CZE",
        "DK":"DNK","EG":"EGY","EE":"EST","FI":"FIN","FR":"FRA","DE":"DEU",
        "GR":"GRC","HU":"HUN","IN":"IND","ID":"IDN","IR":"IRN","IE":"IRL",
        "IL":"ISR","IT":"ITA","JP":"JPN","KZ":"KAZ","KE":"KEN","KR":"KOR",
        "LV":"LVA","LT":"LTU","MX":"MEX","NL":"NLD","NZ":"NZL","NG":"NGA",
        "NO":"NOR","PK":"PAK","PE":"PER","PH":"PHL","PL":"POL","PT":"PRT",
        "RO":"ROU","RU":"RUS","SA":"SAU","RS":"SRB","SG":"SGP","SK":"SVK",
        "SI":"SVN","ZA":"ZAF","ES":"ESP","SE":"SWE","CH":"CHE","TW":"TWN",
        "TR":"TUR","UA":"UKR","GB":"GBR","US":"USA","UY":"URY","VN":"VNM",
    }
    counts["iso_a3"] = counts["country_code"].map(A2_TO_A3)
    print("pycountry not found — using built-in alpha-2→3 lookup.")
    print("  pip install pycountry  for full coverage.")

unmatched = counts[counts["iso_a3"].isna()]["country_code"].tolist()
if unmatched:
    print(f"  Could not map to alpha-3: {unmatched}")

# single merge at the end
world = world.merge(counts[["iso_a3", "count"]], on="iso_a3", how="left")
print(f"Matched {counts['iso_a3'].notna().sum()} / {len(counts)} countries onto world geometry")

Matched 39 / 39 countries onto world geometry


## 4. Plot & save

In [6]:
fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor("white")

# countries with no data
world[world["count"].isna()].plot(
    ax=ax,
    color="#e0e0e0",
    edgecolor="white",
    linewidth=0.4,
)

# countries with data — choropleth
world[world["count"].notna()].plot(
    column="count",
    ax=ax,
    cmap=CMAP,
    edgecolor="white",
    linewidth=0.4,
    legend=True,
    legend_kwds={
        "label": "Author-institution rows",
        "orientation": "horizontal",
        "fraction": 0.03,
        "pad": 0.02,
        "shrink": 0.5,
    },
)

ax.set_title(TITLE, fontsize=14, fontweight="bold", pad=12)
# hide ticks/spines but keep the axes patch so the sea color renders
ax.set_xticks([])
ax.set_yticks([])
for _spine in ax.spines.values():
    _spine.set_visible(False)
ax.set_facecolor("#cce5ff") 

# annotate top-5 countries with count labels
top5 = counts.nlargest(5, "count")
top5_geo = world[world["iso_a3"].isin(top5["iso_a3"].dropna())].copy()
top5_geo = top5_geo.merge(counts[["iso_a3", "count"]], on="iso_a3")

plt.tight_layout()
plt.savefig(OUTPUT_FILE, dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())
print(f"Saved → {OUTPUT_FILE}")
plt.show()

Saved → figures/geo_map.png
